In [1]:
import networkx as nx
import numpy as np
import cvxpy as cp
from scipy.linalg import eigh
import matplotlib.pyplot as plt

In [2]:
global_seed = 0
global_rng = np.random.default_rng(global_seed)

In [3]:
def calc_MH_weights(G):
    n = G.number_of_nodes()
    W = np.zeros((n, n))
    for i in range(n):
        for j in G.adj[i]:
            if(i != j):
                W[i,j] = 1/(1+max(G.degree[i], G.degree[j]))
    np.fill_diagonal(W, np.ones(n) - (W @ np.ones(n)))
    return W

In [4]:
def sample_until(gen, b, max_attempts=1000):
    for t in range(max_attempts):
        g = gen(t)
        if nx.node_connectivity(g) >= b + 1:
            return g
    return None

def create_graph(num_nodes, b, gtype="regular", deg=6, seed=global_seed):
    G = None
    match gtype:
        case "regular":
            G = sample_until(lambda t: nx.random_regular_graph(deg, num_nodes, seed=seed + t), b)
              
        case "ring":
            G = nx.cycle_graph(num_nodes)

        case "erdos":
            p = min(1.0, 3.0 * np.log(num_nodes) / num_nodes)
            G = sample_until(lambda t: nx.gnp_random_graph(num_nodes, p, seed=seed + t), b)
            
        case "hypercube":
            G = nx.convert_node_labels_to_integers(nx.hypercube_graph(k))

        case "complete":
            G = nx.complete_graph(n)
    
    if G == None:
        print(f"Could not generate a {gtype} graph with κ(G)>{b}")
        return None
        
    W = calc_MH_weights(G)
    return G, W

In [5]:
def add_graph_plot(G, B, axis):
    # pos = nx.spring_layout(G)
    pos = nx.kamada_kawai_layout(G)
    colors = ['red' if n in set(B) else 'lightblue' for n in G.nodes()]
    nx.draw(G, pos, node_color=colors, with_labels=True, ax=axis)

In [11]:
def calc_graph_metrics(G,W):
    n = W.shape[0]
    
    L = -W.copy()
    np.fill_diagonal(L, 0.0)
    np.fill_diagonal(L, -L.sum(1))

    mu2 = float(np.sort(np.linalg.eigvalsh(L))[1])
    D_b = float(np.sort(W - np.diag(np.diag(W)), axis=1)[:, -b:].sum(1).max())
    spectral_gap = float(np.abs(np.linalg.eigvalsh(W - np.ones((n, n)) / n)).max())
    
    return {'mu2' : mu2, 'D_b' : D_b, 'spectral_gap': spectral_gap}

In [7]:
# Parameters
num_nodes = 20
b = 2

In [8]:
G,W = create_graph(num_nodes,b,gtype="erdos")

In [9]:
B = global_rng.choice(num_nodes, size=b, replace=False)
H = np.array(list(set(np.arange(num_nodes)) - set(B)))

In [10]:
fig, ax = plt.subplots(1,1,figsize=(8,8))
add_graph_plot(G,B,ax)
plt.close()

In [12]:
calc_graph_metrics(G,W)

{'mu2': 0.21349859388150766,
 'D_b': 0.30952380952380953,
 'spectral_gap': 0.7865014061184923}

In [62]:
"""
min_W D_b(W)  s.t.  nu_b(W) >= nu,  supp(W) = E   (G d-regular, so ||w_i||_0 <= d is free)

Master LP over x = [w | u | s | t], cuts generated by randomly sampling B.
"""
import numpy as np
import scipy.sparse as sp
from scipy.optimize import linprog


def build_master(n, E, b):
    """Static part of the master LP (everything except cuts)."""
    m = len(E)
    nbr = [[] for _ in range(n)]
    for k, (i, j) in enumerate(E):
        nbr[i].append((j, k))
        nbr[j].append((i, k))

    # u index per directed pair (i,j)
    pairs, uidx = [], {}
    for i in range(n):
        for (j, k) in nbr[i]:
            uidx[(i, j)] = len(pairs)
            pairs.append((i, j, k))
    P = len(pairs)                      # = 2m
    N = m + P + n + 1
    iu = lambda p: m + p
    isv = lambda i: m + P + i
    it = m + P + n

    rows, cols, vals, rhs, r = [], [], [], [], 0

    # (1)  b*s_i + sum_j u_ij - t <= 0        [epigraph of D_b]
    for i in range(n):
        rows.append(r); cols.append(isv(i)); vals.append(float(b))
        for (j, k) in nbr[i]:
            rows.append(r); cols.append(iu(uidx[(i, j)])); vals.append(1.0)
        rows.append(r); cols.append(it); vals.append(-1.0)
        rhs.append(0.0); r += 1

    # (2)  w_ij - s_i - u_ij <= 0             [u_ij >= (w_ij - s_i)_+]
    for p, (i, j, k) in enumerate(pairs):
        rows += [r, r, r]; cols += [k, isv(i), iu(p)]; vals += [1.0, -1.0, -1.0]
        rhs.append(0.0); r += 1

    # (3)  sum_{j~i} w_ij <= 1                [w_ii >= 0]
    for i in range(n):
        for (j, k) in nbr[i]:
            rows.append(r); cols.append(k); vals.append(1.0)
        rhs.append(1.0); r += 1

    A = sp.coo_matrix((vals, (rows, cols)), shape=(r, N)).tocsr()
    c = np.zeros(N); c[it] = 1.0
    bounds = [(0, 1)] * m + [(0, None)] * P + [(0, 1)] * n + [(0, 1)]
    return A, np.array(rhs), c, bounds, m, N


def oracle(n, E, w, b, nu, rng, samples, z_prev=None, eps=0.5):
    """Randomly sample B; return violated cuts as (mu2, coef over edges, z)."""
    I = np.array([e[0] for e in E]); J = np.array([e[1] for e in E])
    Aw = sp.coo_matrix((w, (I, J)), shape=(n, n))
    Aw = (Aw + Aw.T).tocsr()

    p = None
    if z_prev is not None:                      # bias toward high Fiedler mass
        p = z_prev ** 2 + eps / n
        p /= p.sum()

    idx = np.arange(n)
    cuts = []
    for _ in range(samples):
        B = rng.choice(n, size=b, replace=False, p=p)
        keep = np.setdiff1d(idx, B)
        As = Aw[keep][:, keep].toarray()
        L = np.diag(As.sum(1)) - As
        ev, evec = np.linalg.eigh(L)            # -> lobpcg for large n
        mu2, z = ev[1], evec[:, 1]
        if mu2 < nu - 1e-9:
            zf = np.zeros(n); zf[keep] = z
            inB = np.zeros(n, bool); inB[B] = True
            coef = np.where(inB[I] | inB[J], 0.0, (zf[I] - zf[J]) ** 2)
            cuts.append((mu2, coef, zf))
    return cuts


def solve(n, E, b, nu, rounds=40, samples=100, per_round=20, seed=0, verbose=True):
    A, rhs, c, bounds, m, N = build_master(n, E, b)
    rng = np.random.default_rng(seed)
    C, z_prev, w = [], None, np.zeros(m)

    for r in range(rounds):
        if C:
            Cm = np.array(C)
            Acut = sp.hstack([sp.csr_matrix(-Cm), sp.csr_matrix((len(C), N - m))])
            Aall = sp.vstack([A, Acut]).tocsr()
            ball = np.concatenate([rhs, -nu * np.ones(len(C))])
        else:
            Aall, ball = A, rhs

        res = linprog(c, A_ub=Aall, b_ub=ball, bounds=bounds, method="highs")
        if not res.success:
            raise RuntimeError(f"LP failed: {res.message}")     # infeasible => nu too large
        w = res.x[:m]

        cuts = oracle(n, E, w, b, nu, rng, samples, z_prev)
        if verbose:
            worst = min(c_[0] for c_ in cuts) if cuts else np.inf
            print(f"round {r:3d}  D_b = {res.fun:.5f}  cuts = {len(C):4d}  "
                  f"worst mu2 = {worst:.5f}  violated = {len(cuts)}/{samples}")
        if not cuts:
            break

        cuts.sort(key=lambda t: t[0])
        z_prev = cuts[0][2]
        C.extend(cc[1] for cc in cuts[:per_round])

    return w, res.fun, len(C)


def random_regular(n, d, seed=0):
    import networkx as nx
    G = nx.random_regular_graph(d, n, seed=seed)
    return [tuple(sorted(e)) for e in G.edges()]


if __name__ == "__main__":
    n, d, b = 60, 14, 3
    E = random_regular(n, d, seed=1)
    lam2 = np.sort(np.linalg.eigvalsh(
        sp.coo_matrix((np.ones(len(E)), ([e[0] for e in E], [e[1] for e in E])),
                      shape=(n, n)).toarray() +
        sp.coo_matrix((np.ones(len(E)), ([e[1] for e in E], [e[0] for e in E])),
                      shape=(n, n)).toarray()))[-2]
    nu_ref = (d - b - lam2) / (d + 1)           # nu_b(W_unif), heuristic
    nu = 0.7 * nu_ref
    print(f"n={n} d={d} b={b}  lam2={lam2:.3f}  nu_ref={nu_ref:.4f}  nu={nu:.4f}\n")

    w, val, ncuts = solve(n, E, b, nu, rounds=40, samples=80)
    print(f"\nD_b = {val:.5f}   (uniform-W closed form: "
          f"{b * nu / (d - lam2 - b):.5f})   cuts used: {ncuts}")

n=60 d=14 b=3  lam2=5.945  nu_ref=0.3370  nu=0.2359

round   0  D_b = 0.00000  cuts =    0  worst mu2 = 0.00000  violated = 80/80
round   1  D_b = 0.05898  cuts =   20  worst mu2 = -0.00000  violated = 80/80
round   2  D_b = 0.07260  cuts =   40  worst mu2 = 0.03827  violated = 80/80
round   3  D_b = 0.07260  cuts =   60  worst mu2 = -0.00000  violated = 80/80


RuntimeError: LP failed: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)